<a href="https://colab.research.google.com/github/alexandrequirinodemelo/alexandrequirino/blob/main/script_refatorado_final_v14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ==============================================================================
# 1. INSTALAÇÃO DE DEPENDÊNCIAS E BIBLIOTECAS
# ==============================================================================
# SCRIPT ATUALIZADO NO DIA 18/05/2026 COM OS CHECK LIST REVISADOS

!pip install -q keras-tuner

import pandas as pd
import numpy as np
import os
import warnings
import shutil
import logging
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from prophet import Prophet
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

# ==============================================================================
# 2. CONFIGURAÇÕES GLOBAIS E CONTROLE DE ABLAÇÃO
# ==============================================================================
USAR_CHUVA         = True  # Altere para True ou False para rodar os testes de ablação
MAX_LAG            = 5
ANO_ALVO_PREVISAO  = 2025
ANO_LIMITE_INFERIOR= 2005
GRADE_DEPTH        = [2, 3, 4, 5, 6, 8, 10]

# ==============================================================================
# 3. PIPELINE DE DADOS UNIFICADO (SEM VAZAMENTO METODOLÓGICO)
# ==============================================================================
def carregar_e_estruturar_dados():
    if os.path.exists("produtividade_novo_encolding.xlsx"):
        df_matriz = pd.read_excel("produtividade_novo_encolding.xlsx")
    else:
        df_matriz = pd.read_csv("produtividade_novo_encolding.xlsx - productivity.csv")

    col_ano = df_matriz.columns[0]
    df_longo = df_matriz.melt(id_vars=[col_ano], var_name='estado', value_name='produtividade')
    df_longo.rename(columns={col_ano: 'ano_int'}, inplace=True)
    df_longo['estado'] = df_longo['estado'].astype(str).str.strip().str.upper()

    if os.path.exists("produtividade_novo_completo_2025_26.xlsx"):
        df_gab_raw = pd.read_excel("produtividade_novo_completo_2025_26.xlsx")
    else:
        df_gab_raw = pd.read_csv("produtividade_novo_completo_2025_26.xlsx - 2012_a_2025.csv")

    df_gab_raw.columns = [str(c).strip().lower() for c in df_gab_raw.columns]
    df_gab_raw['ano_resolvido'] = df_gab_raw['ano'].astype(str).apply(lambda x: 2025 if '2025' in x else None)

    df_gab_2025 = df_gab_raw[(df_gab_raw['ano_resolvido'] == ANO_ALVO_PREVISAO) & (df_gab_raw['levantamento'] == 4)]
    df_gab_2025 = df_gab_2025[['estado', 'produtividade']].rename(columns={'estado': 'estado', 'produtividade': 'produtividade_real'})
    df_gab_2025['estado'] = df_gab_2025['estado'].astype(str).str.strip().str.upper()
    df_gab_2025 = df_gab_2025.drop_duplicates(subset=['estado']).reset_index(drop=True)

    if ANO_ALVO_PREVISAO not in df_longo['ano_int'].unique():
        estados = df_longo['estado'].unique()
        df_2025 = pd.DataFrame({'ano_int': [ANO_ALVO_PREVISAO]*len(estados), 'estado': estados, 'produtividade': [np.nan]*len(estados)})
        df_base = pd.concat([df_longo, df_2025])
    else:
        df_base = df_longo.copy()

    df_base = df_base.sort_values(['estado', 'ano_int']).reset_index(drop=True)

    for i in range(1, MAX_LAG + 1):
        df_base[f'Prod_t_{i}'] = df_base.groupby('estado')['produtividade'].shift(i)

    if os.path.exists("chuva_consolidada_2005_2025_corrigida.xlsx"):
        df_chuva = pd.read_excel("chuva_consolidada_2005_2025_corrigida.xlsx")
    else:
        df_chuva = pd.read_csv("chuva_consolidada_2005_2025_corrigida.xlsx - Sheet1.csv")

    df_chuva.columns = [str(c).strip().lower() for c in df_chuva.columns]
    df_chuva.rename(columns={'uf': 'estado', 'ano': 'ano_int'}, inplace=True)
    df_chuva['estado'] = df_chuva['estado'].astype(str).str.strip().str.upper()

    df_chuva_agrupada = df_chuva.groupby(['estado', 'ano_int'])[['chuva_q1_mm', 'chuva_q2_mm', 'chuva_q3_mm']].mean().reset_index()
    df_chuva_agrupada.columns = ['estado', 'ano_int', 'Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']

    df_base = pd.merge(df_base, df_chuva_agrupada, on=['estado', 'ano_int'], how='left')
    df_base[['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']] = df_base[['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']].fillna(0.0)

    if not USAR_CHUVA:
        df_base['Chuva_Q1_mm'], df_base['Chuva_Q2_mm'], df_base['Chuva_Q3_mm'] = 0.0, 0.0, 0.0

    le = LabelEncoder()
    df_base['estado_id'] = le.fit_transform(df_base['estado'])

    df_dummies = pd.get_dummies(df_base['estado'], prefix='est', dtype=int)
    df_base = pd.concat([df_base, df_dummies], axis=1)

    df_base = pd.merge(df_base, df_gab_2025, on='estado', how='left')
    return df_base, le

# ==============================================================================
# 4. ARQUITETURA NEURAL ADAPTATIVA (CORREÇÃO DO GRAFO DE ABLAÇÃO)
# ==============================================================================
class DeepAgroDissertacao(kt.HyperModel):
    def __init__(self, n_est, n_lags_input, usar_clima, r_type='LSTM'):
        self.n_est = n_est
        self.n_lags = n_lags_input
        self.usar_clima = usar_clima
        self.r_type = r_type

    def build(self, hp):
        in_est = layers.Input(shape=(1,), name="input_est")
        in_seq = layers.Input(shape=(self.n_lags, 4 if self.usar_clima else 1), name="input_seq")

        emb = layers.Flatten()(layers.Embedding(self.n_est, hp.Int('emb', 4, 16))(in_est))

        if self.r_type == 'LSTM':
            rnn_out = layers.LSTM(units=hp.Int('u', 32, 128, step=32), return_sequences=False)(in_seq)
        elif self.r_type == 'LSTM-GRU':
            x_rnn = layers.LSTM(units=hp.Int('u_lstm', 32, 128, step=32), return_sequences=True)(in_seq)
            rnn_out = layers.GRU(units=hp.Int('u_gru', 16, 64, step=16), return_sequences=False)(x_rnn)

        # Ajuste Crítico: Conexão dinâmica de inputs e listas baseada na ablação climática
        if self.usar_clima:
            in_cli = layers.Input(shape=(3,), name="input_cli")
            concat = layers.Concatenate()([rnn_out, emb, in_cli])
            inputs_lista = [in_est, in_seq, in_cli]
        else:
            concat = layers.Concatenate()([rnn_out, emb])
            inputs_lista = [in_est, in_seq]

        x = layers.Dense(hp.Int('d', 16, 64, step=16), activation='relu')(concat)
        x = layers.Dropout(0.2)(x)
        out = layers.Dense(1)(x)

        model = keras.Model(inputs=inputs_lista, outputs=out)
        model.compile(optimizer=keras.optimizers.Adam(hp.Choice('lr', [1e-2, 1e-3])), loss='mse')
        return model

def construir_tensores_neural(df, sc_lags, sc_clima, n_lags, usar_clima):
    clima_cols = ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']
    lags_scaled = sc_lags.transform(df[[f'Prod_t_{i}' for i in range(1, MAX_LAG + 1)]])[:, :n_lags]
    clima_scaled = sc_clima.transform(df[clima_cols])

    N = len(df)
    dim_features = 4 if usar_clima else 1
    in_seq = np.zeros((N, n_lags, dim_features))

    for t in range(n_lags):
        in_seq[:, t, 0] = lags_scaled[:, t]
        if usar_clima:
            in_seq[:, t, 1] = clima_scaled[:, 0]
            in_seq[:, t, 2] = clima_scaled[:, 1]
            in_seq[:, t, 3] = clima_scaled[:, 2]

    if usar_clima:
        return [df['estado_id'].values.reshape(-1, 1), in_seq, clima_scaled]
    else:
        return [df['estado_id'].values.reshape(-1, 1), in_seq]

def otimizar_tabular_passado(df_historico, mod):
    ano_val = ANO_ALVO_PREVISAO - 1
    df_train_val = df_historico[(df_historico['ano_int'] < ano_val) & (df_historico['produtividade'] > 0)]
    df_test_val  = df_historico[df_historico['ano_int'] == ano_val]
    if len(df_test_val) == 0: return 2, 4

    best_l, best_d, best_mae = 2, 4, float('inf')
    col_dummies = [c for c in df_historico.columns if c.startswith('est_')]

    for l in range(1, MAX_LAG + 1):
        features_temporarias = [f'Prod_t_{i}' for i in range(1, l + 1)]
        df_tr_v_clean = df_train_val.dropna(subset=features_temporarias)
        df_ts_v_clean = df_test_val.dropna(subset=features_temporarias)
        if len(df_tr_v_clean) == 0 or len(df_ts_v_clean) == 0: continue

        for d in GRADE_DEPTH:
            f_tab = col_dummies + features_temporarias + ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']
            X_tr_v, X_ts_v = df_tr_v_clean[f_tab], df_ts_v_clean[f_tab]

            if mod == 'XGBoost': m_t = XGBRegressor(max_depth=d, random_state=42).fit(X_tr_v, df_tr_v_clean['produtividade'])
            elif mod == 'Random Forest': m_t = RandomForestRegressor(max_depth=d, random_state=42, n_jobs=-1).fit(X_tr_v, df_tr_v_clean['produtividade'])
            else: m_t = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_tr_v, df_tr_v_clean['produtividade'])

            p_v = m_t.predict(X_ts_v)
            mae_v = mean_absolute_error(df_ts_v_clean['produtividade'], p_v)
            if mae_v < best_mae: best_mae, best_l, best_d = mae_v, l, d

    return best_l, best_d

# ==============================================================================
# 5. EXECUÇÃO DO PIPELINE E CONTROLE DE EXPORTAÇÃO
# ==============================================================================
df_base, label_encoder = carregar_e_estruturar_dados()
n_estados_total = len(label_encoder.classes_)

resumo_metricas = []
resumo_hparams = []

colunas_max_lags = [f'Prod_t_{i}' for i in range(1, MAX_LAG + 1)]
colunas_clima = ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']
colunas_estados_dummies = [c for c in df_base.columns if c.startswith('est_')]

df_train = df_base[(df_base['ano_int'] < ANO_ALVO_PREVISAO) & (df_base['produtividade'] > 0)].dropna(subset=colunas_max_lags).reset_index(drop=True)
df_test  = df_base[df_base['ano_int'] == ANO_ALVO_PREVISAO].copy()

df_res_2025 = df_test[['ano_int', 'estado', 'produtividade_real']].rename(columns={'produtividade_real': 'Real_2025'})

sc_lags = StandardScaler().fit(df_train[colunas_max_lags])
sc_clima = StandardScaler().fit(df_train[colunas_clima])
sc_y = StandardScaler().fit(df_train[['produtividade']])
y_train_scaled = sc_y.transform(df_train[['produtividade']]).flatten()

modelos_lista = ['LSTM', 'LSTM-GRU', 'XGBoost', 'Random Forest', 'Decision Tree', 'Prophet', 'SNaive']

for mod in modelos_lista:
    print(f"-> Computando: {mod}")

    if mod in ['LSTM', 'LSTM-GRU']:
        lags_fixos_neural = 3
        inputs_train = construir_tensores_neural(df_train, sc_lags, sc_clima, lags_fixos_neural, USAR_CHUVA)
        inputs_test = construir_tensores_neural(df_test, sc_lags, sc_clima, lags_fixos_neural, USAR_CHUVA)

        dir_tuner = f't_dir_{mod.lower().replace("-","_")}_auditoria'
        if os.path.exists(dir_tuner): shutil.rmtree(dir_tuner)

        tuner = kt.RandomSearch(
            DeepAgroDissertacao(n_est=n_estados_total, n_lags_input=lags_fixos_neural, usar_clima=USAR_CHUVA, r_type=mod),
            objective=kt.Objective('val_loss', direction='min'), max_trials=3, directory=dir_tuner
        )

        es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
        tuner.search(inputs_train, y_train_scaled, epochs=50, validation_split=0.15, callbacks=[es], verbose=0)

        m = tuner.get_best_models(1)[0]
        hp_b = tuner.get_best_hyperparameters(1)[0]
        p = sc_y.inverse_transform(m.predict(inputs_test, verbose=0)).flatten()

        if mod == 'LSTM-GRU':
            config_str = f"Híbrido (LSTM_Units: {hp_b.get('u_lstm')}, GRU_Units: {hp_b.get('u_gru')})"
        else:
            config_str = f"Rede {mod} (Units: {hp_b.get('u')}, Dense: {hp_b.get('d')})"
        res_lag = lags_fixos_neural
        shutil.rmtree(dir_tuner)

    elif mod in ['XGBoost', 'Random Forest', 'Decision Tree']:
        best_l, best_d = otimizar_tabular_passado(df_base, mod)
        f_tab = colunas_estados_dummies + [f'Prod_t_{i}' for i in range(1, best_l + 1)] + colunas_clima

        if mod == 'XGBoost': m_t = XGBRegressor(max_depth=best_d, random_state=42).fit(df_train[f_tab], df_train['produtividade'])
        elif mod == 'Random Forest': m_t = RandomForestRegressor(max_depth=best_d, random_state=42, n_jobs=-1).fit(df_train[f_tab], df_train['produtividade'])
        else: m_t = DecisionTreeRegressor(max_depth=best_d, random_state=42).fit(df_train[f_tab], df_train['produtividade'])

        p = m_t.predict(df_test[f_tab])
        res_lag, config_str = best_l, f"OHE Geográfico, Lag: {best_l}, Depth: {best_d}"

    elif mod == 'Prophet':
        dict_p = {}
        for e in df_test['estado'].unique():
            df_e = df_train[df_train['estado'] == e][['ano_int', 'produtividade'] + colunas_clima].rename(columns={'ano_int': 'ds', 'produtividade': 'y'})
            df_e['ds'] = pd.to_datetime(df_e['ds'], format='%Y')
            m_prophet = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
            m_prophet.fit(df_e)
            df_fut = pd.DataFrame({'ds': [pd.to_datetime(ANO_ALVO_PREVISAO, format='%Y')]})
            for c in colunas_clima: df_fut[c] = 0.0
            dict_p[e] = m_prophet.predict(df_fut)['yhat'].values[0]
        p = df_test['estado'].map(dict_p).values
        res_lag, config_str = 0, "Série Temporal Univariada"

    elif mod == 'SNaive':
        p = df_test['Prod_t_1'].values
        res_lag, config_str = 1, "Lag 1 Fixo Baseline"

    df_res_2025[f'Prev_{mod}'] = np.maximum(p, 0)

    for idx, row in df_res_2025.iterrows():
        est_nome = row['estado']
        val_real = row['Real_2025']
        val_prev = row[f'Prev_{mod}']
        if pd.isna(val_real) or pd.isna(val_prev): continue

        rmse_local = np.abs(val_real - val_prev)
        mae_local  = np.abs(val_real - val_prev)
        mape_local = (np.abs(val_real - val_prev) / (val_real if val_real != 0 else 1)) * 100

        resumo_metricas.append({
            'Estado': est_nome, 'Modelo': mod,
            'RMSE': rmse_local, 'MAE': mae_local, 'MAPE (%)': mape_local, 'R2': np.nan
        })

    resumo_hparams.append({
        'Modelo': mod, 'Lag_Utilizado': res_lag, 'Configuracao_Final': config_str
    })

df_tabela_metricas = pd.DataFrame(resumo_metricas).drop_duplicates(subset=['Estado', 'Modelo']).sort_values(by=['Estado', 'Modelo']).reset_index(drop=True)
df_tabela_hparams = pd.DataFrame(resumo_hparams)

# ==============================================================================
# 6. EXPORTAÇÃO EXCEL AUDITADA
# ==============================================================================
suffix = "SEM_CHUVA" if not USAR_CHUVA else "COM_CHUVA"
nome_arquivo = f"RESULTADOS_FINAIS_{suffix}.xlsx"
with pd.ExcelWriter(nome_arquivo) as writer:
    df_res_2025.to_excel(writer, sheet_name='Previsoes_2025', index=False)
    df_tabela_metricas.to_excel(writer, sheet_name='Metricas', index=False)
    df_tabela_hparams.to_excel(writer, sheet_name='Hiperparametros', index=False)

print(f"\n>>> PIPELINE COMPLETADO COM SUCESSO! Arquivo salvo em: '{nome_arquivo}'")
try:
    from google.colab import files
    files.download(nome_arquivo)
except:
    pass

-> Computando: LSTM
-> Computando: LSTM-GRU
-> Computando: XGBoost
-> Computando: Random Forest
-> Computando: Decision Tree
-> Computando: Prophet
-> Computando: SNaive

>>> PIPELINE COMPLETADO COM SUCESSO! Arquivo salvo em: 'RESULTADOS_FINAIS_COM_CHUVA.xlsx'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>